# Destination-Adjusted Player Projection

Interactive companion to `src/portalpoint/modeling/destination_projection.py` /
`scripts/run_destination_projection.py`, matching the script/notebook-in-sync pattern used
by all other models in this repo.

All fit/score/write logic lives in the `destination_projection` module — this notebook is
for delta-model benchmarking, coverage audits, and sensitivity checks the script itself
doesn't expose (tier-matrix heat maps, per-delta contribution distributions, CI calibration
curves).

Full design/contract: `docs/models/destination_projection_plan.md`.
Production rerun: `scripts/run_destination_projection.py` (handles MLflow + portal-scope filtering).

**Model version:** `player-destination-proj-v1`

**Prerequisites before running this notebook:**
1. `scripts/run_player_projection.py --phase cross-season` → fills `player-proj-phase2a-fcast-v1` rows
2. `scripts/run_playing_time.py --target-season 2027` → fills `playing_time_projections` for 2027

Without step 2, all inference rows will be dropped at the PT hard gate (Cell 4 will warn).

In [ ]:
# Cell 0 — Imports + Config
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from portalpoint.modeling import destination_projection as dp
from portalpoint.modeling.io import get_sync_engine

MODEL_VERSION = dp.MODEL_VERSION
SOURCE_SEASON = 2026      # last completed season — player stats / portal candidates sourced here
TARGET_SEASON = 2027      # season being projected into
TRAIN_SEASONS = [2022, 2023, 2024, 2025, 2026]  # historical transfer outcomes for delta models

engine = get_sync_engine()
print(f"Model version: {MODEL_VERSION}")
print(f"Source season: {SOURCE_SEASON}  Target season: {TARGET_SEASON}")
print(f"Train seasons: {TRAIN_SEASONS}")

## 1. Coverage Audit

Confirms that the prerequisite neutral projections and playing-time projections exist for the
target season before running a long inference pass.

In [ ]:
# Cell 1 — Coverage audit
from sqlalchemy import text

with engine.connect() as conn:
    neutral_counts = conn.execute(text("""
        SELECT model_version, season, count(*)
        FROM player_projections
        WHERE school_id IS NULL AND projection_mode = 'neutral'
        GROUP BY 1, 2
        ORDER BY 2 DESC, 1
    """)).fetchall()

    pt_counts = conn.execute(text("""
        SELECT model_version, season, count(*) AS n_rows,
               count(DISTINCT player_id) AS n_players,
               count(DISTINCT school_id) AS n_schools
        FROM playing_time_projections
        GROUP BY 1, 2
        ORDER BY 2 DESC, 1
    """)).fetchall()

    dest_counts = conn.execute(text("""
        SELECT model_version, season, count(*) AS n_rows,
               count(DISTINCT player_id) AS n_players,
               count(DISTINCT school_id) AS n_schools
        FROM player_projections
        WHERE school_id IS NOT NULL AND projection_mode = 'destination'
        GROUP BY 1, 2
        ORDER BY 2 DESC, 1
    """)).fetchall()

    portal_count = conn.execute(text("""
        SELECT count(DISTINCT player_id)
        FROM player_team_fit_scores
        WHERE season = :season AND is_portal_candidate = true
    """), {"season": SOURCE_SEASON}).scalar()

print("--- Neutral projections (school_id IS NULL) ---")
for version, season, n in neutral_counts:
    marker = " <- priority" if version == dp.NEUTRAL_MODEL_PRIORITY[0] else ""
    print(f"  {version:35s} season={season} n={n:,}{marker}")

print(f"\n--- Playing time projections (target season={TARGET_SEASON}) ---")
if pt_counts:
    for version, season, nr, np_, ns in pt_counts:
        print(f"  {version:30s} season={season} rows={nr:,} players={np_:,} schools={ns:,}")
else:
    print("  EMPTY — run scripts/run_playing_time.py --target-season 2027 first")

print(f"\n--- Destination projections already written (model_version={MODEL_VERSION}) ---")
if dest_counts:
    for version, season, nr, np_, ns in dest_counts:
        print(f"  {version:30s} season={season} rows={nr:,} players={np_:,} schools={ns:,}")
else:
    print("  None yet — will be populated by this notebook's Cell 9 or run_destination_projection.py")

print(f"\nPortal candidates (is_portal_candidate=True, season={SOURCE_SEASON}): {portal_count:,}")

## 2. Load Portal Candidates + Neutral Projections

In [ ]:
# Cell 2 — Load portal candidates + neutral projections
candidate_ids = dp.load_portal_candidates(engine, fit_context_season=SOURCE_SEASON)
neutral_df = dp.load_neutral_projections(engine, candidate_ids)

print(f"Portal candidates: {len(candidate_ids):,}")
print(f"Neutral projections found: {len(neutral_df):,}")
print(f"Coverage: {len(neutral_df)/max(len(candidate_ids),1):.1%} of candidates have a neutral projection")

if not neutral_df.empty:
    print("\nNeutral model version distribution:")
    display(neutral_df["neutral_model_version"].value_counts().rename("count").to_frame())
    print(f"\nValue per 100 — mean={neutral_df['value_per_100'].mean():.3f} "
          f"std={neutral_df['value_per_100'].std():.3f} "
          f"min={neutral_df['value_per_100'].min():.3f} "
          f"max={neutral_df['value_per_100'].max():.3f}")

## 3. Load Playing Time Projections (Hard Gate)

The inner join on `playing_time_projections` is the hard gate — pairs with no PT estimate are
excluded from destination rows. This cell shows coverage before and after the gate so any
PT-model gap is visible early.

In [ ]:
# Cell 3 — Playing time hard gate
pt_df = dp.load_playing_time_projections(engine, candidate_ids, TARGET_SEASON)

if pt_df.empty:
    print("WARNING: playing_time_projections is empty for season", TARGET_SEASON)
    print("Run: uv run python scripts/run_playing_time.py --target-season", TARGET_SEASON)
    print("\nThe notebook can still validate delta models (Cells 4-6) but Cell 7+")
    print("inference and Cell 9 write will produce empty frames.")
else:
    print(f"PT rows: {len(pt_df):,}")
    print(f"PT unique players: {pt_df['player_id'].nunique():,}")
    print(f"PT unique schools: {pt_df['school_id'].nunique():,}")
    print(f"After PT inner join (estimate): {pt_df['player_id'].isin(neutral_df['player_id'].values).sum():,} rows")

    print(f"\nExpected minutes — mean={pt_df['expected_minutes'].mean():.1f} "
          f"std={pt_df['expected_minutes'].std():.1f}")
    print(f"Expected usage — mean={pt_df['expected_usage'].mean():.4f} "
          f"std={pt_df['expected_usage'].std():.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    pt_df["expected_minutes"].plot(kind="hist", bins=30, ax=ax[0], title="Expected minutes")
    pt_df["expected_usage"].plot(kind="hist", bins=30, ax=ax[1], title="Expected usage")
    plt.tight_layout()
    plt.show()

## 4. Train Historical Delta Models

Fits the role/usage Ridge model and builds the competition-tier transition matrix from
historical transfer outcomes (TRAIN_SEASONS). The style/skill and roster-context deltas
are rule-based and have no training step.

In [ ]:
# Cell 4 — Load historical transfer outcomes + fit delta models
training_df = dp.load_historical_transfer_outcomes(engine, TRAIN_SEASONS)
training_df = dp.build_destination_training_examples(training_df)

print(f"Training rows: {len(training_df):,}")
if not training_df.empty:
    print(f"  Seasons: {sorted(training_df['dest_season'].unique().tolist())}")
    print(f"  value_delta mean={training_df['value_delta'].mean():.3f} std={training_df['value_delta'].std():.3f}")
    print(f"  usage_delta mean={training_df['usage_delta'].mean():.4f} std={training_df['usage_delta'].std():.4f}")

# Fit role/usage Ridge model
role_model, role_scaler, role_feature_names, role_residual_std = dp.fit_role_usage_model(training_df)
print(f"\nRole/usage model: n_features={len(role_feature_names)}, residual_std={role_residual_std:.3f}")

if role_model is not None:
    coef_df = pd.Series(
        role_model.coef_, index=role_feature_names, name="coefficient"
    ).sort_values(key=abs, ascending=False)
    print("\nTop 10 coefficients (role/usage model):")
    display(coef_df.head(10).to_frame())

In [ ]:
# Cell 4b — Competition tier transition matrix
tier_mean_mat, tier_std_mat = dp.build_competition_tier_matrix(training_df)

print("Tier transition mean delta (row=source_tier, col=dest_tier):")
display(tier_mean_mat.round(3))

print("\nTier transition residual std:")
display(tier_std_mat.round(3))

# Heat map
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(tier_mean_mat.values, cmap="RdYlGn", vmin=-0.5, vmax=0.5)
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(["Tier 1\n(High-Major)", "Tier 2", "Tier 3", "Tier 4\n(Low-Major)"])
ax.set_yticklabels(["Tier 1\n(Source)", "Tier 2", "Tier 3", "Tier 4"])
ax.set_xlabel("Destination tier")
ax.set_ylabel("Source tier")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{tier_mean_mat.values[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, label="Mean value delta")
plt.title("Competition tier transition matrix")
plt.tight_layout()
plt.show()

## 5. Build Inference Frame

Joins neutral projections, PT projections, team context, pairwise fit context, and roster
state into one flat frame. Requires PT rows to exist (Cell 3 gate).

In [ ]:
# Cell 5 — Build inference frame
team_context_df = dp.load_destination_team_context(engine, SOURCE_SEASON)
fit_context_df = dp.load_pairwise_fit_context(engine, candidate_ids, SOURCE_SEASON)
roster_df = dp.load_roster_state_features(engine, SOURCE_SEASON)
source_stats_df = dp.load_source_player_stats(engine, candidate_ids, SOURCE_SEASON)

print(f"Team context: {len(team_context_df):,} schools")
print(f"Pairwise fit rows: {len(fit_context_df):,}")
print(f"Roster state features: {len(roster_df):,} school-seasons")
print(f"Source player stats: {len(source_stats_df):,} players")

if not pt_df.empty:
    frame = dp.build_destination_inference_frame(
        neutral_df=neutral_df,
        pt_df=pt_df,
        team_context_df=team_context_df,
        fit_context_df=fit_context_df,
        roster_df=roster_df,
        source_stats_df=source_stats_df,
        target_season=TARGET_SEASON,
        source_season=SOURCE_SEASON,
    )
    print(f"\nInference frame: {len(frame):,} rows ({frame['player_id'].nunique():,} players "
          f"× {frame['school_id'].nunique():,} schools)")
    display(frame[["player_id", "school_id", "value_per_100", "expected_minutes", "expected_usage",
                   "gap_match", "source_tier", "dest_tier"]].head(5))
else:
    frame = pd.DataFrame()
    print("Skipping inference frame (no PT rows)")

## 6. Delta Computation + Cap Enforcement

Applies all four delta components and enforces ±0.75/±1.50 caps.

In [ ]:
# Cell 6 — Delta computation
if not frame.empty:
    frame["role_usage_delta"] = dp.compute_role_usage_delta(
        frame, role_model, role_scaler, role_feature_names
    )
    frame["style_skill_fit_delta"] = dp.compute_style_skill_fit_delta(frame)
    frame["roster_context_delta"] = dp.compute_roster_context_delta(frame)
    frame["competition_level_delta"] = dp.compute_competition_level_delta(
        frame, tier_mean_mat, tier_std_mat
    )
    frame = dp.apply_delta_caps(frame)

    delta_cols = ["role_usage_delta", "style_skill_fit_delta", "roster_context_delta",
                  "competition_level_delta", "total_context_delta"]
    print("Delta summary stats:")
    display(frame[delta_cols].describe().round(4))

    fig, axes = plt.subplots(1, 5, figsize=(16, 3))
    for ax, col in zip(axes, delta_cols):
        frame[col].plot(kind="hist", bins=30, ax=ax, title=col.replace("_delta", ""))
    plt.tight_layout()
    plt.show()
    
    # Total delta vs neutral value scatter
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(frame["value_per_100"], frame["total_context_delta"], alpha=0.05, s=5)
    ax.axhline(0, color="red", linewidth=0.8)
    ax.set_xlabel("Neutral value per 100")
    ax.set_ylabel("Total context delta")
    ax.set_title("Context delta vs. neutral value")
    plt.tight_layout()
    plt.show()
else:
    print("No frame to compute deltas on")

## 7. Final Value + Uncertainty + Rate Translation

In [ ]:
# Cell 7 — Final destination-adjusted value + uncertainty
if not frame.empty:
    frame = dp.translate_neutral_to_destination_value(frame)

    # CI calibration (uses labeled training rows as proxy)
    ci_scale = 1.0
    if not training_df.empty and "dest_total_rapm" in training_df.columns:
        train_pred = training_df[["dest_total_rapm", "neutral_value"]].copy()
        train_pred["destination_value_per_100"] = train_pred["neutral_value"]
        train_pred["value_ci_lower"] = train_pred["neutral_value"] - 2.0
        train_pred["value_ci_upper"] = train_pred["neutral_value"] + 2.0
        ci_scale = dp.calibrate_ci_scale(train_pred, actual_col="dest_total_rapm")

    print(f"CI calibration scale: {ci_scale:.3f}")

    frame = dp.propagate_destination_uncertainty(frame, role_residual_std, ci_scale)
    frame = dp.translate_rates_to_destination_stats(frame, team_context_df)

    print(f"\nDestination value summary:")
    print(f"  mean={frame['destination_value_per_100'].mean():.3f} "
          f"std={frame['destination_value_per_100'].std():.3f}")
    print(f"  neutral mean={frame['value_per_100'].mean():.3f} "
          f"(delta={frame['destination_value_per_100'].mean()-frame['value_per_100'].mean():.3f})")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(frame["value_per_100"], bins=40, alpha=0.5, label="neutral")
    ax.hist(frame["destination_value_per_100"], bins=40, alpha=0.5, label="destination-adjusted")
    ax.set_xlabel("Value per 100 possessions")
    ax.legend()
    ax.set_title("Neutral vs. destination-adjusted value distribution")
    plt.tight_layout()
    plt.show()
else:
    print("No frame to score")

## 8. Spot Check: Top Portal Candidates for a Sample School

Simulates the program-facing query: `school_id=X` → ranked portal candidates.

In [ ]:
# Cell 8 — Spot check: top-N portal candidates for a sample school
SAMPLE_SCHOOL_ID = 1  # change to any school_id in your DB

if not frame.empty and "school_id" in frame.columns:
    school_frame = frame[frame["school_id"] == SAMPLE_SCHOOL_ID].copy()
    school_frame = school_frame.sort_values("destination_value_per_100", ascending=False)

    print(f"Top 10 portal candidates for school_id={SAMPLE_SCHOOL_ID}:")
    cols = [
        "player_id", "value_per_100", "destination_value_per_100",
        "total_context_delta", "role_usage_delta", "style_skill_fit_delta",
        "roster_context_delta", "competition_level_delta",
        "expected_minutes", "gap_match",
    ]
    display(school_frame[cols].head(10).round(3))

    # Delta contribution stacked bar for top-10
    top10 = school_frame.head(10)
    delta_components = ["role_usage_delta", "style_skill_fit_delta",
                        "roster_context_delta", "competition_level_delta"]
    fig, ax = plt.subplots(figsize=(10, 4))
    bottom = np.zeros(len(top10))
    colors = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c"]
    labels = ["Role/Usage", "Style/Skill", "Roster Context", "Competition"]
    for col, color, label in zip(delta_components, colors, labels):
        vals = top10[col].values
        pos = np.maximum(vals, 0)
        neg = np.minimum(vals, 0)
        ax.bar(range(len(top10)), pos, bottom=np.maximum(bottom, 0), color=color, label=label, alpha=0.8)
        ax.bar(range(len(top10)), neg, bottom=np.minimum(bottom, 0), color=color, alpha=0.8)
        bottom = bottom + vals
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(range(len(top10)))
    ax.set_xticklabels(top10["player_id"].astype(str).str[-6:], rotation=45, fontsize=8)
    ax.set_ylabel("Context delta (value per 100)")
    ax.set_title(f"Delta decomposition — top-10 candidates for school_id={SAMPLE_SCHOOL_ID}")
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print(f"No rows for school_id={SAMPLE_SCHOOL_ID} (or frame is empty)")

## 9. DB Write (gated — off by default)

Production writes belong in `scripts/run_destination_projection.py` (it owns MLflow tracking).
This cell exists for one-off manual spot-checks only — flip `SHOULD_WRITE` and narrow `SCHOOL_ID_FILTER`
before ever running it for real.

In [ ]:
# Cell 9 — Gated DB write
SHOULD_WRITE = False
DRY_RUN = True   # True = compute records but skip upsert; False = real write
SCHOOL_ID_FILTER = None  # None = all schools; [10, 20] = restrict to these school_ids

if SHOULD_WRITE and not frame.empty:
    write_frame = frame if SCHOOL_ID_FILTER is None else frame[frame["school_id"].isin(SCHOOL_ID_FILTER)]
    explanation = dp.build_explanation_payload(write_frame, SOURCE_SEASON, TARGET_SEASON)
    records = dp.build_destination_projection_records(write_frame, explanation, SOURCE_SEASON, TARGET_SEASON)
    if DRY_RUN:
        print(f"Dry run: would write {len(records):,} destination projection rows")
        print(f"Example record (first):")
        print(dict(zip(
            ["player_id", "school_id", "season", "projection_mode",
             "value_per_100", "value_ci_lower", "value_ci_upper",
             "projected_minutes", "projected_usage",
             "projected_box_score", "projected_rates",
             "skill_states", "skill_percentiles", "uncertainty",
             "explanation", "model_version", "computed_at", "expires_at"],
            records[0]
        )))
    else:
        n_written = dp.upsert_destination_projections(engine, records)
        print(f"Wrote {n_written:,} destination projection rows (model_version={MODEL_VERSION})")
else:
    print("SHOULD_WRITE is False — no DB write performed.")
    print("Use scripts/run_destination_projection.py for production runs.")